In [26]:
import pandas as pd
from transformers import T5Tokenizer , Trainer ,TrainingArguments ,T5ForConditionalGeneration

In [27]:
train_data = pd.read_csv(r"C:\Users\jiten\OneDrive\Desktop\python\mini_project\mini project nlp\samsum-train.csv")
val_data = pd.read_csv(r"C:\Users\jiten\OneDrive\Desktop\python\mini_project\mini project nlp\samsum-validation.csv")

In [28]:
#randoom sampling
train_data = train_data.sample(n = 4000 , random_state=42).reset_index(drop =True)
val_data = val_data.sample(n = 500 , random_state=42).reset_index(drop= True)

In [29]:
# data pre-processing

import re
def clean_data(text):
    text = re.sub(r"\r\n" , " " , text) # line
    text = re.sub(r"\s+" , " " , text) # spaces
    text = re.sub(r"<.*?>" , " " , text) # html tags
    text = text.strip().lower()
    return text

In [30]:
train_data["dialogue"]  = train_data["dialogue"].apply(clean_data)
train_data["summary"]  = train_data["summary"].apply(clean_data)

val_data["dialogue"]  = val_data["dialogue"].apply(clean_data)
val_data["summary"]  = val_data["summary"].apply(clean_data)



In [31]:
#Tokenzation
tokenizer = T5Tokenizer.from_pretrained("t5-small")

#raw data => tokenized inputs for fine-tuning

def tokenize(example):
    inputs = tokenizer(
        example["dialogue"],
        padding="max_length",
        max_length=512,
        truncation=True
    )

    targets = tokenizer(
        example["summary"],
        padding="max_length",
        max_length=150,
        truncation=True
    )

    inputs["labels"] = targets["input_ids"]
    return inputs

In [32]:
train_dataset = train_data.apply(tokenize, axis=1).tolist()
val_dataset = val_data.apply(tokenize, axis=1).tolist()

In [33]:
train_dataset[0]

{'input_ids': [25208, 10, 7102, 55, 3, 23, 764, 640, 48, 403, 17, 77, 31, 7, 1108, 11, 3, 23, 816, 24, 25, 429, 253, 34, 1477, 25208, 10, 3, 7997, 15, 10, 7102, 55, 3, 10, 61, 2049, 6, 68, 3, 23, 31, 162, 641, 608, 34, 5, 3, 10, 61, 3, 7997, 15, 10, 68, 2049, 21, 1631, 81, 140, 3, 10, 61, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0,

In [34]:
#working with model
#nlp => genetraion task

model = T5ForConditionalGeneration.from_pretrained('t5-small')

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 8684.55it/s]


In [35]:
import torch

if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

print("device :" ,device)
model.to(device)


device : cpu


T5ForConditionalGeneration(
  (shared): Embedding(32128, 512)
  (encoder): T5Stack(
    (embed_tokens): Embedding(32128, 512)
    (block): ModuleList(
      (0): T5Block(
        (layer): ModuleList(
          (0): T5LayerSelfAttention(
            (SelfAttention): T5Attention(
              (q): Linear(in_features=512, out_features=512, bias=False)
              (k): Linear(in_features=512, out_features=512, bias=False)
              (v): Linear(in_features=512, out_features=512, bias=False)
              (o): Linear(in_features=512, out_features=512, bias=False)
              (relative_attention_bias): Embedding(32, 8)
            )
            (layer_norm): T5LayerNorm()
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (1): T5LayerFF(
            (DenseReluDense): T5DenseActDense(
              (wi): Linear(in_features=512, out_features=2048, bias=False)
              (wo): Linear(in_features=2048, out_features=512, bias=False)
              (dropout): Drop

In [36]:
# training arguments
training_args = TrainingArguments(
    output_dir= "./results",

    num_train_epochs= 5,
    weight_decay=  0.01,

    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,

    eval_strategy= "epoch",
    save_strategy= "epoch",

    warmup_steps= 500
    # 0=> r defaults
)

In [37]:
trainer = Trainer(
    model = model,
    args = training_args,
    train_dataset= train_dataset,
    eval_dataset= val_dataset
)

In [38]:
# train the model
#trainer.train()

In [39]:
model.save_pretrained("./saved_summary_model")
tokenizer.save_pretrained("./saved_summary_model")


Writing model shards: 100%|██████████| 1/1 [00:00<00:00,  7.06it/s]


('./saved_summary_model\\tokenizer_config.json',
 './saved_summary_model\\tokenizer.json')

In [40]:
model = T5ForConditionalGeneration.from_pretrained("./saved_summary_model")
tokenizer = T5Tokenizer.from_pretrained("./saved_summary_model")

Loading weights: 100%|██████████| 131/131 [00:00<00:00, 8733.41it/s]


In [52]:
#Test the core logic for summarization
def summarize_dialogue(dialogue):
    dialogue = clean_data(dialogue) # clean

    # token 
    inputs = tokenizer(dialogue,
                        padding = "max_length",
                        max_length = 512,
                        truncation = True,
                        return_tensors = "pt" 
                        ).to(device)

    #generate the summary
    model.to(device)
    targets = model.generate(

        input_ids = inputs["input_ids"],
        attention_mask = inputs["attention_mask"],
        max_length = 150,
        num_beams  = 4,
        early_stopping = True

    )


    # token ids conert to summary => decoding
    summary = tokenizer.decode(targets[0],skip_special_tokens=True)
    return summary

In [55]:
test_dialogue = """
Reporter: In today's technology news, artificial intelligence continues to expand rapidly across industries, from healthcare to finance and education. Recent reports suggest that AI adoption has significantly increased over the past few years.

Reporter: Companies are investing heavily in machine learning systems to automate tasks, improve decision-making, and enhance customer experiences. However, this growth has also raised questions about job displacement and ethical concerns.

Expert: AI systems are becoming more capable due to advances in deep learning and access to large datasets. These models can now perform complex tasks such as language understanding, image recognition, and even code generation.

Expert: At the same time, there are valid concerns about bias in AI models, as they often reflect the data they are trained on. Ensuring fairness and transparency is becoming a key area of research.

Reporter: Governments and organizations are beginning to introduce regulations to guide the development and deployment of AI technologies. The goal is to balance innovation with accountability.

Expert: Another challenge is explainability. Many modern AI systems, especially deep neural networks, operate as “black boxes,” making it difficult to understand how decisions are made.

Reporter: Experts also highlight the importance of responsible AI development, including data privacy, security, and long-term societal impact.

Expert: Looking ahead, collaboration between researchers, policymakers, and industry leaders will be crucial to ensure that AI systems are developed and used in a safe and beneficial way.


"""
summary = summarize_dialogue(test_dialogue)

In [56]:
print("summary" , summary)

summary experts highlight the importance of responsible ai development, including data privacy, security, and long-term societal impact. ensuring fairness and transparency is becoming a key area of research.
